<a href="https://colab.research.google.com/github/nesbertharuna/SMS-SMISHING-PROJECT/blob/main/notebooks/smishing_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SMS Smishing Detection — Google Colab

Run the **research ML pipeline** from this repo on Colab:

1. Get the code (GitHub clone, Drive, or ZIP upload)
2. Install Python dependencies
3. Train TF-IDF + Logistic Regression on Dataset1
4. Evaluate on the held-out test split
5. Classify example SMS messages interactively
6. *(Optional)* Compare ML vs rule-based baseline

**Note:** The Next.js web app and React Native mobile app are not run here — only the Python ML/research stack.

Repo: [nesbertharuna/SMS-SMISHING-PROJECT](https://github.com/nesbertharuna/SMS-SMISHING-PROJECT)

## 1. Choose how to load the project

Set `SETUP_MODE` below:
- `"github"` — clone from GitHub *(recommended; push your latest code first)*
- `"drive"` — use a copy already on Google Drive
- `"upload"` — upload a ZIP of the project folder in the next cell

In [ ]:
SETUP_MODE = "github"  # "github" | "drive" | "upload"

GITHUB_REPO = "https://github.com/nesbertharuna/SMS-SMISHING-PROJECT.git"
GITHUB_BRANCH = "main"
REPO_DIR_NAME = "SMS-SMISHING-PROJECT"

# Only used when SETUP_MODE == "drive"
DRIVE_REPO_PATH = "/content/drive/MyDrive/SMS SMISHING SOFTWARE"

# Only used when SETUP_MODE == "upload"
UPLOAD_ZIP_PATH = "/content/smishing_project.zip"

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

CONTENT = Path("/content")

if SETUP_MODE == "github":
    target = CONTENT / REPO_DIR_NAME
    if target.exists():
        shutil.rmtree(target)
    subprocess.run(
        ["git", "clone", "--branch", GITHUB_BRANCH, "--depth", "1", GITHUB_REPO, str(target)],
        check=True,
    )
    REPO = target

elif SETUP_MODE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    REPO = Path(DRIVE_REPO_PATH)
    if not REPO.exists():
        raise FileNotFoundError(f"Drive path not found: {REPO}")

elif SETUP_MODE == "upload":
    from google.colab import files

    uploaded = files.upload()  # pick your project ZIP
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    zip_name = next(iter(uploaded))
    zip_path = CONTENT / zip_name
    extract_to = CONTENT / "uploaded_repo"
    if extract_to.exists():
        shutil.rmtree(extract_to)
    extract_to.mkdir(parents=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    # Use the single top-level folder if the zip contains one
    children = [p for p in extract_to.iterdir() if p.name != "__MACOSX"]
    REPO = children[0] if len(children) == 1 and children[0].is_dir() else extract_to

else:
    raise ValueError(f"Unknown SETUP_MODE: {SETUP_MODE!r}")

os.chdir(REPO)
print("Working directory:", REPO.resolve())
print("Contents:", [p.name for p in sorted(REPO.iterdir())[:12]], "...")

## 2. Install dependencies

In [ ]:
!pip install -q -r ml-backend/requirements.txt

## 3. Train and evaluate (full pipeline)

Runs: normalize Dataset1 → stratified splits → train → test metrics.

`--eval-overwrite` writes fixed report filenames (`reports/metrics_ml_test.json`) for easy loading below.

In [12]:
!{sys.executable} scripts/train_full_pipeline.py --dataset dataset1 --eval-overwrite


> /usr/bin/python3 /content/SMS-SMISHING-PROJECT/scripts/load_dataset1.py --in /content/SMS-SMISHING-PROJECT/ml-backend/dataset/Dataset1.csv --out /content/SMS-SMISHING-PROJECT/data/processed/dataset1.csv
Wrote 20 rows -> /content/SMS-SMISHING-PROJECT/data/processed/dataset1.csv
label
benign(0)      10
smishing(1)    10
Name: count, dtype: int64

> /usr/bin/python3 /content/SMS-SMISHING-PROJECT/scripts/make_splits.py --in /content/SMS-SMISHING-PROJECT/data/processed/dataset1.csv --out_dir /content/SMS-SMISHING-PROJECT/data/splits --seed 42
train: n=14 | label=0: 7, label=1: 7
val: n=3 | label=0: 1, label=1: 2
test: n=3 | label=0: 2, label=1: 1
Wrote splits -> /content/SMS-SMISHING-PROJECT/data/splits

> /usr/bin/python3 /content/SMS-SMISHING-PROJECT/experiments/train.py --train data/splits/train.csv --val data/splits/val.csv --artifact ml-backend/artifacts/pipeline_lr_tfidf.joblib --seed 42
Saved model -> /content/SMS-SMISHING-PROJECT/ml-backend/artifacts/pipeline_lr_tfidf.joblib
Save

In [ ]:
import json
from IPython.display import display
import pandas as pd

metrics_path = REPO / "reports" / "metrics_ml_test.json"
if not metrics_path.exists():
    # fallback: newest timestamped metrics file
    candidates = sorted((REPO / "reports").glob("metrics_ml_test_*.json"))
    if not candidates:
        raise FileNotFoundError("No metrics file found under reports/")
    metrics_path = candidates[-1]

metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print(f"Loaded: {metrics_path.relative_to(REPO)}")
display(pd.DataFrame([metrics.get("metrics_pos1", metrics)]))

## 4. Classify SMS messages

Uses the same inference path as the web app (`ml-backend/api/classify.py`).

In [ ]:
import json
import subprocess

CLASSIFY_SCRIPT = REPO / "ml-backend" / "api" / "classify.py"
ARTIFACT = "ml-backend/artifacts/pipeline_lr_tfidf.joblib"


def classify_sms(text: str, top_k: int = 5) -> dict:
    payload = json.dumps({"text": text, "top_k": top_k, "artifact": ARTIFACT})
    proc = subprocess.run(
        [sys.executable, str(CLASSIFY_SCRIPT)],
        input=payload,
        capture_output=True,
        text=True,
        cwd=REPO,
        check=False,
    )
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr or proc.stdout)
    return json.loads(proc.stdout)


def print_result(result: dict) -> None:
    print("Label:", result["label"].upper())
    if result.get("risk_percent_smishing") is not None:
        print("Smishing score:", result["risk_percent_smishing"], "%")
    print(result.get("verdict_plain", ""))
    for line in result.get("signals_toward_smishing", []):
        print("  •", line)
    for line in result.get("signals_toward_benign", []):
        print("  •", line)

In [ ]:
examples = [
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: Your bank account is locked. Verify now at http://bit.ly/verify-acct",
    "Your package could not be delivered. Pay $2.99 fee: https://track-parcel.xyz",
]

for msg in examples:
    print("=" * 72)
    print("SMS:", msg)
    print_result(classify_sms(msg))
    print()

In [ ]:
# Try your own message
my_sms = "Type your SMS here"
print_result(classify_sms(my_sms))

## 5. (Optional) ML vs rule-based baseline

In [13]:
!{sys.executable} experiments/evaluate_rules.py
!{sys.executable} experiments/compare_ml_vs_rules.py

Rules test metrics (pos_label=1): {'accuracy': 0.6666666666666666, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
Wrote predictions -> /content/SMS-SMISHING-PROJECT/reports/pred_rules_test.csv
Wrote -> /content/SMS-SMISHING-PROJECT/reports/metrics_rules_test.json
ML: {'accuracy': 0.6666666666666666, 'precision_pos1': 0.0, 'recall_pos1': 0.0, 'f1_pos1': 0.0}
Rules: {'accuracy': 0.6666666666666666, 'precision_pos1': 0.0, 'recall_pos1': 0.0, 'f1_pos1': 0.0}
McNemar: {'b_ml_correct_rules_wrong': 0, 'c_ml_wrong_rules_correct': 0, 'n_discordant': 0, 'p_value': 1.0}
Wrote -> /content/SMS-SMISHING-PROJECT/reports/compare_ml_vs_rules.json


In [14]:
compare_path = REPO / "reports" / "compare_ml_vs_rules.json"
if compare_path.exists():
    compare = json.loads(compare_path.read_text(encoding="utf-8"))
    display(pd.DataFrame(compare["metrics"]).T)
    print("McNemar:", compare.get("mcnemar_exact"))
else:
    print("Comparison file not found — run the cell above first.")

,accuracy,precision_pos1,recall_pos1,f1_pos1
ml,0.666667,0.0,0.0,0.0
rules,0.666667,0.0,0.0,0.0


McNemar: {'b_ml_correct_rules_wrong': 0, 'c_ml_wrong_rules_correct': 0, 'n_discordant': 0, 'p_value': 1.0}


## 6. Save artifacts to Google Drive (optional)

Keeps the trained model and reports after the Colab session ends.

In [ ]:
SAVE_TO_DRIVE = False  # set True to copy artifacts to Drive

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/smishing_colab_outputs")
    dest.mkdir(parents=True, exist_ok=True)
    for rel in [
        "ml-backend/artifacts/pipeline_lr_tfidf.joblib",
        "ml-backend/artifacts/pipeline_lr_tfidf.meta.json",
        "reports/metrics_ml_test.json",
        "reports/pred_ml_test.csv",
    ]:
        src = REPO / rel
        if src.exists():
            shutil.copy2(src, dest / src.name)
            print("Saved", dest / src.name)
    print("Done. Artifacts in", dest)